# HMM Event Detection — RxInfer

2-state HMM (quiet / active) with Gaussian emissions on the x-axis magnetometer signal. The Viterbi path gives us the event segmentation directly.

- State 1: **quiet** — low variance, stable mean
- State 2: **active** — higher variance, shifted mean (water running)

Train emission parameters from labeled calibration data, then run on live stream.

In [43]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.add(["HTTP", "JSON3"])

  Activating project at `~/Documents/GitHub/RxInferExamples.jl/examples/Flow Project/Flow Fingerprint`
   Resolving package versions...
     Project No packages added to or removed from `~/Documents/GitHub/RxInferExamples.jl/examples/Flow Project/Flow Fingerprint/Project.toml`
    Manifest No packages added to or removed from `~/Documents/GitHub/RxInferExamples.jl/examples/Flow Project/Flow Fingerprint/Manifest.toml`


In [44]:
using RxInfer
using HTTP, JSON3
using Dates, Statistics
using Plots
gr()

Plots.GRBackend()

In [45]:
const HASURA_URL = "https://hasura.pipestuesday.org/v1/graphql"
const HASURA_SECRET = "PIPE_SUPERMMMSECRET_PIPE"

function gql(query::String; variables=nothing)
    body = Dict{String, Any}("query" => query)
    if !isnothing(variables)
        body["variables"] = variables
    end
    resp = HTTP.post(HASURA_URL,
        ["Content-Type" => "application/json", "x-hasura-admin-secret" => HASURA_SECRET],
        JSON3.write(body))
    data = JSON3.read(String(resp.body))
    haskey(data, :errors) && error("GraphQL: $(data[:errors])")
    return data[:data]
end

println("Hasura client ready")

Hasura client ready


## 1. Fetch Calibration Labels + Learn Emission Parameters

Get labeled events, fetch mag data for each, and compute the Gaussian emission parameters (mean, variance) for quiet vs active states.

In [46]:
SENSOR_ID = 6
PADDING_S = 5  # seconds of context before/after each label

# Fetch all calibration labels
labels_raw = gql("""
query {
  calibration_label(order_by: {created_at: asc}) {
    id, start_time, end_time
    fixture { id, name, type, sensor_id }
  }
}
""")[:calibration_label]

# Filter to our sensor
labels = filter(labels_raw) do lab
    fix = get(lab, :fixture, nothing)
    !isnothing(fix) && get(fix, :sensor_id, nothing) == SENSOR_ID
end

println("Calibration labels for sensor $SENSOR_ID: $(length(labels))")

Calibration labels for sensor 6: 14


In [47]:
# Fetch mag data for each label and collect quiet/active samples
quiet_samples = Float64[]
active_samples = Float64[]

for lab in labels
    t0 = DateTime(string(lab.start_time)[1:19], dateformat"yyyy-mm-ddTHH:MM:SS")
    t1 = DateTime(string(lab.end_time)[1:19], dateformat"yyyy-mm-ddTHH:MM:SS")
    
    fetch_start = t0 - Second(PADDING_S)
    fetch_end = t1 + Second(PADDING_S)
    
    mag = gql("""
    query GetMag(\$sid: bigint!, \$since: timestamptz!, \$until: timestamptz!) {
      mag_report(
        where: {sensor_id: {_eq: \$sid}, created_at: {_gte: \$since, _lte: \$until}}
        order_by: {created_at: asc}, limit: 100000
      ) { created_at, x_axis_reading }
    }
    """, variables=Dict(
        "sid" => string(SENSOR_ID),
        "since" => string(fetch_start) * "Z",
        "until" => string(fetch_end) * "Z"
    ))[:mag_report]
    
    length(mag) == 0 && continue
    
    for row in mag
        x = parse(Float64, string(row.x_axis_reading))
        rt = DateTime(string(row.created_at)[1:19], dateformat"yyyy-mm-ddTHH:MM:SS")
        if rt >= t0 && rt <= t1
            push!(active_samples, x)
        else
            push!(quiet_samples, x)
        end
    end
end

println("Quiet samples: $(length(quiet_samples))")
println("Active samples: $(length(active_samples))")

# Emission parameters
μ_quiet = mean(quiet_samples)
σ²_quiet = var(quiet_samples)
μ_active = mean(active_samples)
σ²_active = var(active_samples)

println("\nLearned emissions:")
println("  Quiet:  μ=$(round(μ_quiet, digits=2)), σ²=$(round(σ²_quiet, digits=2))")
println("  Active: μ=$(round(μ_active, digits=2)), σ²=$(round(σ²_active, digits=2))")

Quiet samples: 1177
Active samples: 3263

Learned emissions:
  Quiet:  μ=-636.7, σ²=9933.89
  Active: μ=-632.86, σ²=9901.54


## 2. Define 2-State HMM in RxInfer

Simple HMM: hidden state switches between quiet (1) and active (2), each with Gaussian emissions. Transition matrix is learned; emission means/precisions use informed priors from the labeled data.

In [48]:
const N_STATES = 2

@model function event_hmm(y, prior_A)
    # Transition matrix: each row is a Dirichlet
    A ~ DirichletCollection(prior_A)
    
    # Emission means — use data-informed priors
    local m
    local w
    for k in 1:N_STATES
        m[k] ~ NormalMeanPrecision(0.0, 0.001)
        w[k] ~ GammaShapeRate(1.0, 1.0)
    end

    s_prev ~ Categorical(fill(1.0 / N_STATES, N_STATES))

    for t in eachindex(y)
        s[t] ~ DiscreteTransition(s_prev, A)
        y[t] ~ NormalMixture(switch = s[t], m = m, p = w)
        s_prev = s[t]
    end
end

@constraints function event_hmm_constraints()
    q(s_prev, s, A, m, w) = q(s_prev, s)q(A)q(m)q(w)
end

println("Model defined — 2-state HMM with Gaussian emissions")

Model defined — 2-state HMM with Gaussian emissions


## 3. Fetch Live Stream + Run Inference

In [49]:
MINUTES_BACK = 15

# Fetch last N minutes (or last available window)
now_utc = Dates.now(Dates.UTC)
since = now_utc - Minute(MINUTES_BACK)

mag = gql("""
query GetMag(\$sid: bigint!, \$since: timestamptz!) {
  mag_report(
    where: {sensor_id: {_eq: \$sid}, created_at: {_gte: \$since}}
    order_by: {created_at: asc}, limit: 100000
  ) { created_at, x_axis_reading }
}
""", variables=Dict("sid" => string(SENSOR_ID), "since" => string(since) * "Z"))[:mag_report]

if length(mag) == 0
    println("No recent data — fetching last available window...")
    latest = gql("""
    query Latest(\$sid: bigint!) {
      mag_report(where: {sensor_id: {_eq: \$sid}}, order_by: {created_at: desc}, limit: 1) { created_at }
    }
    """, variables=Dict("sid" => string(SENSOR_ID)))[:mag_report]
    
    end_time = DateTime(string(latest[1].created_at)[1:19], dateformat"yyyy-mm-ddTHH:MM:SS")
    start_time = end_time - Minute(MINUTES_BACK)
    
    mag = gql("""
    query GetMag(\$sid: bigint!, \$since: timestamptz!, \$until: timestamptz!) {
      mag_report(
        where: {sensor_id: {_eq: \$sid}, created_at: {_gte: \$since, _lte: \$until}}
        order_by: {created_at: asc}, limit: 100000
      ) { created_at, x_axis_reading }
    }
    """, variables=Dict(
        "sid" => string(SENSOR_ID),
        "since" => string(start_time) * "Z",
        "until" => string(end_time) * "Z"
    ))[:mag_report]
end

# Parse
timestamps = DateTime[]
x_values = Float64[]
for row in mag
    push!(timestamps, DateTime(string(row.created_at)[1:19], dateformat"yyyy-mm-ddTHH:MM:SS"))
    push!(x_values, parse(Float64, string(row.x_axis_reading)))
end

println("$(length(x_values)) readings: $(timestamps[1]) → $(timestamps[end])")

8838 readings: 2026-03-29T22:30:16 → 2026-03-29T22:45:09


In [50]:
# Build transition prior from labeled data
# Quiet state is sticky (high self-transition), active is moderately sticky
# prior_A[i,j] = pseudo-counts for transitioning from state i to state j
prior_A = [
    50.0  1.0;   # quiet → quiet (sticky), quiet → active (rare)
     1.0 20.0    # active → quiet, active → active (moderately sticky)
]

# Initialize emission posteriors from labeled data
init = @initialization begin
    q(m) = [NormalMeanPrecision(μ_quiet, 0.1), NormalMeanPrecision(μ_active, 0.1)]
    q(w) = [GammaShapeRate(2.0, σ²_quiet), GammaShapeRate(2.0, σ²_active)]
    q(s) = Categorical(fill(1.0 / N_STATES, N_STATES))
end

println("Running inference on $(length(x_values)) observations...")
@time result = infer(
    model          = event_hmm(prior_A = prior_A),
    data           = (y = x_values,),
    constraints    = event_hmm_constraints(),
    initialization = init,
    returnvars     = (s = KeepLast(), m = KeepLast(), w = KeepLast(), A = KeepLast()),
    iterations     = 30,
    free_energy    = true,
    options        = (limit_stack_depth = 500,),
)

println("Done! Final free energy: $(round(result.free_energy[end], digits=2))")

Running inference on 8838 observations...


ErrorException: The factorization around `NormalMixture` must be the naive mean-field.

## 4. Extract State Sequence + Visualize

In [51]:
# Get MAP state at each timestamp
state_posteriors = result.posteriors[:s]
states = [argmax(probvec(sp)) for sp in state_posteriors]

# Figure out which state is "active" — the one with higher emission variance
m_post = [mean(result.posteriors[:m][k]) for k in 1:N_STATES]
w_post = [mean(result.posteriors[:w][k]) for k in 1:N_STATES]

println("Learned emissions:")
for k in 1:N_STATES
    println("  State $k: μ=$(round(m_post[k], digits=2)), precision=$(round(w_post[k], digits=4)) (σ²=$(round(1/w_post[k], digits=2)))")
end

# Active state = one with lower precision (higher variance) or more different mean
active_state = argmin(w_post)
println("\nActive state: $active_state")

is_active = states .== active_state
n_active = sum(is_active)
println("Active timestamps: $n_active / $(length(states)) ($(round(100*n_active/length(states), digits=1))%)")

UndefVarError: UndefVarError: `result` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [52]:
# Extract event regions (contiguous active blocks) with 3s padding
PAD_S = 3
total_s = (timestamps[end] - timestamps[1]).value / 1000.0
sps = length(timestamps) / total_s  # samples per second
pad_n = round(Int, PAD_S * sps)

events = Tuple{Int,Int}[]
in_ev = false
ev_start = 0
for i in eachindex(is_active)
    if is_active[i] && !in_ev
        ev_start = i
        in_ev = true
    elseif !is_active[i] && in_ev
        push!(events, (ev_start, i - 1))
        in_ev = false
    end
end
in_ev && push!(events, (ev_start, length(is_active)))

# Pad
events_padded = [(max(1, s - pad_n), min(length(timestamps), e + pad_n)) for (s, e) in events]

println("Detected $(length(events_padded)) event(s):")
for (i, (s, e)) in enumerate(events_padded)
    dur = (timestamps[e] - timestamps[s]).value / 1000.0
    println("  Event $i: $(timestamps[s]) → $(timestamps[e]) ($(round(dur, digits=1))s)")
end

UndefVarError: UndefVarError: `is_active` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [53]:
# Plot signal with HMM state overlay
p1 = plot(timestamps, x_values, lw=0.4, alpha=0.5, color=:steelblue, label="raw",
          ylabel="X-Axis Reading", title="HMM Event Detection — Sensor $SENSOR_ID")

# Shade events
for (i, (s, e)) in enumerate(events_padded)
    vspan!([timestamps[s], timestamps[e]], alpha=0.2, color=:orange, label=(i==1 ? "event" : ""))
    vline!([timestamps[s]], color=:green, lw=2, alpha=0.8, label=(i==1 ? "onset" : ""))
    vline!([timestamps[e]], color=:red, lw=2, alpha=0.8, label=(i==1 ? "offset" : ""))
end

# State probability plot
active_probs = [probvec(sp)[active_state] for sp in state_posteriors]
p2 = plot(timestamps, active_probs, lw=0.8, color=:darkorange, label="P(active)",
          ylabel="P(active)", xlabel="Time", ylim=(0, 1.05))
hline!([0.5], ls=:dash, color=:gray, alpha=0.5, label="")
for (s, e) in events_padded
    vspan!([timestamps[s], timestamps[e]], alpha=0.15, color=:orange, label="")
end

# Free energy convergence
p3 = plot(result.free_energy, lw=2, color=:purple, xlabel="Iteration", ylabel="Free Energy",
          title="Convergence", legend=false)

l = @layout [a; b; c{0.25h}]
plot(p1, p2, p3, layout=l, size=(1200, 700))

UndefVarError: UndefVarError: `events_padded` not defined in `Main`
Suggestion: check for spelling errors or missing imports.